# 11 — Fault classification with the production posterior (System I, N3, measured inlet)

Replaces the classification parts of the old notebooks 05a and 11. **Production posterior** = notebook 36 (20k, 5-seed ensemble,
measured inlet temperatures as inputs). **Data:** D1 (`observations_measured_inlet.npz`, 300 windows, notebook 02b) and a larger
evaluation set **L1** (1,000 windows per scenario, same generator and inlet distribution as D1, new seeds, not stored) for
standard errors of about 0.01. The **naive** posterior (nb31, nominal inlet assumed) appears only in the sensitivity section.

**Rules (fixed before running; runbook §2f row b).**
* Class = quadrant of (α, β) with threshold 0.90; predicted class = the quadrant holding most of the posterior mass.
* **Three headline numbers per scenario:** class accuracy, β 90% coverage at the truth, β sd (mean posterior sd), with binomial
  standard errors; the headline figures come from L1, D1 is the stored-dataset result.
* **Macro-F1 headline over Sc1–Sc5** (well-specified scenarios); all six scenarios and Sc7 (+2 K sensor drift, misspecified by design)
  are reported separately; per-class F1 and the confusion matrix.
* **Saturation split:** a window is "saturated" if more than 50% of its samples have Qc ≥ 395 L/min (with the inlet varying over ±2 K
  the plant often reaches the jacket limit, notebook 02b); the three numbers per group.
* **Checks (stop if violated):** β coverage at the truth ≥ 0.80 in each of Sc1–Sc4 on L1; overall prior leakage < 5%; everything finite.
* Not here: the LDA baseline and the pre-computed NUTS classification of 05a (the NUTS part waits for the NUTS baseline); the 30-day
  classification timeline of the old 11 moves to notebook 10.


In [1]:
import json, os, sys, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
import importlib, system_I_config, ws_protocols, ws_eval, ws_metrics, ws_inlet, nb31_budget_convergence, nb35_measured_inlet, nb36_production_measured_inlet, ws_phase3
for _m in (system_I_config, ws_protocols, ws_eval, ws_metrics, ws_inlet, nb31_budget_convergence, nb35_measured_inlet, nb36_production_measured_inlet, ws_phase3):
    importlib.reload(_m)
we, wm, P3 = ws_eval, ws_metrics, ws_phase3
T0 = time.time(); R = ROOT / "results"
SCN = {1: "Sc1", 2: "Sc2", 3: "Sc3", 4: "Sc4", 5: "Sc5", 7: "Sc7"}
CLS = list(wm.CLASSES); N_DRAW = 2000
post_pm, prod = P3.load_production_posterior()
print("Production posterior:", prod["production_budget"], "simulations,", "ensemble of %d" % len(prod["members"]), "| summary set", prod["summary_set"], "| eps", prod["inlet_model"]["eps_K"], "K")

def evaluate(post, X, theta2, names, seed=0):
    smp, leak = we.sample_within_prior(post, X, N_DRAW, rng_seed=seed)
    return smp, leak

def three(smp, theta2, names, mask=None):
    """Per scenario: n, class accuracy (+-SE), beta coverage (+-SE), beta sd, beta bias, alpha sd."""
    if mask is None: mask = np.ones(len(theta2), bool)
    yt = wm.class_of_theta(theta2); yp = wm.class_of_samples(smp)
    cov = wm.credible_coverage_at_truth(smp, theta2, 0.90)[:, 1]; sd = smp.std(axis=1); mean = smp.mean(axis=1)
    rows = []
    for s in sorted(set(names)):
        m = (names == s) & mask
        if m.sum() == 0: continue
        n = int(m.sum()); acc = float((yp[m] == yt[m]).mean()); c = float(cov[m].mean())
        rows.append(dict(scenario=s, n=n, class_acc=acc, acc_se=(acc * (1 - acc) / n) ** 0.5, cov90_b=c, cov_se=(c * (1 - c) / n) ** 0.5,
                         sd_b=float(sd[m, 1].mean()), bias_b=float((mean[m, 1] - theta2[m, 1]).mean()), sd_a=float(sd[m, 0].mean())))
    return pd.DataFrame(rows)

def names_of(d): return np.array([SCN[int(i)] for i in d["scenario_id"]])


Production posterior: 20000 simulations, ensemble of 5 | summary set S1m_measured_inlet | eps 0.1 K


## 1. Production posterior on D1 (the stored dataset, 300 windows)

In [2]:
d1 = P3.load_d1(); n1 = names_of(d1); th1 = d1["theta"][:, 2:]
X1 = P3.production_features(d1)
smp1, leak1 = evaluate(post_pm, X1, th1, n1)
tab1 = three(smp1, th1, n1); display(tab1.round(3))
print("prior leakage (mean fraction of raw flow mass outside the prior): %.4f" % leak1.mean())
tab1.to_csv(R / "11_N3_D1_three_numbers.csv", index=False)


,scenario,n,class_acc,acc_se,cov90_b,cov_se,sd_b,bias_b,sd_a
0,Sc1,50,0.98,0.020,0.96,0.028,0.046,0.000,0.055
1,Sc2,50,0.82,0.054,0.94,0.034,0.046,0.007,0.056
2,Sc3,50,0.98,0.020,0.88,0.046,0.042,-0.006,0.029
3,Sc4,50,0.90,0.042,0.96,0.028,0.044,-0.001,0.043
4,Sc5,50,1.00,0.000,0.94,0.034,0.042,0.001,0.059
5,Sc7,50,0.50,0.071,0.00,0.000,0.028,-0.394,0.072


prior leakage (mean fraction of raw flow mass outside the prior): 0.0151


## 2. Larger evaluation set L1 (1,000 windows per scenario): the headline figures

In [3]:
t0 = time.time()
dL = P3.generate_d1(n_rep=1000, seed_base=3_800_000)
nL = names_of(dL); thL = dL["theta"][:, 2:]
XL = P3.production_features(dL)
smpL, leakL = evaluate(post_pm, XL, thL, nL, seed=1)
ytL, ypL = wm.class_of_theta(thL), wm.class_of_samples(smpL)
tabL = three(smpL, thL, nL); display(tabL.round(3))
print(f"L1: {len(thL)} windows simulated and evaluated in {time.time() - t0:.0f}s | prior leakage {leakL.mean():.4f}")
tabL.to_csv(R / "11_N3_L1_three_numbers.csv", index=False)


,scenario,n,class_acc,acc_se,cov90_b,cov_se,sd_b,bias_b,sd_a
0,Sc1,1000,0.970,0.005,0.916,0.009,0.044,-0.003,0.056
1,Sc2,1000,0.857,0.011,0.904,0.009,0.044,-0.002,0.056
2,Sc3,1000,0.992,0.003,0.925,0.008,0.045,0.002,0.029
3,Sc4,1000,0.766,0.013,0.910,0.009,0.045,0.001,0.044
4,Sc5,1000,0.977,0.005,0.927,0.008,0.043,0.004,0.059
5,Sc7,1000,0.341,0.015,0.002,0.001,0.029,-0.395,0.069


L1: 6000 windows simulated and evaluated in 183s | prior leakage 0.0169


## 3. Checks (stop if violated)

In [4]:
assert np.isfinite(smp1).all() and np.isfinite(smpL).all() and np.isfinite(X1).all() and np.isfinite(XL).all()
bad = tabL[tabL.scenario.isin(["Sc1", "Sc2", "Sc3", "Sc4"]) & (tabL.cov90_b < 0.80)]
if len(bad):
    raise RuntimeError("STOP: beta coverage at the truth < 0.80 in " + ", ".join(f"{r.scenario} ({r.cov90_b:.2f})" for r in bad.itertuples()))
if leakL.mean() >= 0.05 or leak1.mean() >= 0.05:
    raise RuntimeError(f"STOP: prior leakage {max(leakL.mean(), leak1.mean()):.3f} >= 0.05")
print("checks passed: beta coverage >= 0.80 in Sc1-Sc4 (L1):", tabL[tabL.scenario.isin(["Sc1", "Sc2", "Sc3", "Sc4"])][["scenario", "cov90_b"]].round(3).values.tolist(),
      "| leakage %.4f / %.4f" % (leak1.mean(), leakL.mean()))


checks passed: beta coverage >= 0.80 in Sc1-Sc4 (L1): [[np.str_('Sc1'), 0.916], [np.str_('Sc2'), 0.904], [np.str_('Sc3'), 0.925], [np.str_('Sc4'), 0.91]] | leakage 0.0151 / 0.0169


## 4. Macro-F1, per-class F1 and the confusion matrix

In [5]:
def per_class(yt, yp):
    rows = []
    for c in CLS:
        tp = int(((yp == c) & (yt == c)).sum()); fp = int(((yp == c) & (yt != c)).sum()); fn = int(((yp != c) & (yt == c)).sum())
        rows.append(dict(cls=c, n_true=int((yt == c).sum()), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), f1=2 * tp / max(2 * tp + fp + fn, 1)))
    return pd.DataFrame(rows)
out = []
for lab, d_th, d_n, d_smp in (("L1 (1,000 per scenario)", thL, nL, smpL), ("D1 (50 per scenario)", th1, n1, smp1)):
    yt, yp = wm.class_of_theta(d_th), wm.class_of_samples(d_smp)
    for grp, m in (("Sc1-Sc5 (headline)", np.isin(d_n, ["Sc1", "Sc2", "Sc3", "Sc4", "Sc5"])), ("all six", np.ones(len(d_n), bool)), ("Sc7 only", d_n == "Sc7")):
        out.append(dict(set=lab, scenarios=grp, n=int(m.sum()), accuracy=float((yp[m] == yt[m]).mean()), macro_f1=wm.macro_f1(yt[m], yp[m])))
f1tab = pd.DataFrame(out); display(f1tab.round(3)); f1tab.to_csv(R / "11_N3_macro_f1.csv", index=False)
m5 = np.isin(nL, ["Sc1", "Sc2", "Sc3", "Sc4", "Sc5"]); ytL5, ypL5 = ytL[m5], ypL[m5]
print("Per-class F1, L1, Sc1-Sc5:"); pc = per_class(ytL5, ypL5); display(pc.round(3)); pc.to_csv(R / "11_N3_per_class_f1.csv", index=False)
cm = pd.crosstab(pd.Series(ytL5, name="true"), pd.Series(ypL5, name="predicted")).reindex(index=CLS, columns=CLS, fill_value=0)
print("Confusion matrix (counts), L1, Sc1-Sc5:"); display(cm); print("row-normalised:"); display((cm.div(cm.sum(1).replace(0, 1), axis=0)).round(3))
cm.to_csv(R / "11_N3_confusion_L1.csv")


,set,scenarios,n,accuracy,macro_f1
0,"L1 (1,000 per scenario)",Sc1-Sc5 (headline),5000,0.912,0.907
1,"L1 (1,000 per scenario)",all six,6000,0.817,0.824
2,"L1 (1,000 per scenario)",Sc7 only,1000,0.341,0.509
3,D1 (50 per scenario),Sc1-Sc5 (headline),250,0.936,0.936
4,D1 (50 per scenario),all six,300,0.863,0.869
5,D1 (50 per scenario),Sc7 only,50,0.500,0.667


Per-class F1, L1, Sc1-Sc5:


,cls,n_true,precision,recall,f1
0,healthy,1000,0.859,0.970,0.911
1,fouling_dominant,2000,0.935,0.917,0.926
2,decay_dominant,1000,0.904,0.992,0.946
3,combined,1000,0.943,0.766,0.845


Confusion matrix (counts), L1, Sc1-Sc5:


predicted,healthy,fouling_dominant,decay_dominant,combined
true,,,,
healthy,970,20,10,0
fouling_dominant,127,1834,1,38
decay_dominant,0,0,992,8
combined,32,108,94,766


row-normalised:


predicted,healthy,fouling_dominant,decay_dominant,combined
true,,,,
healthy,0.970,0.020,0.010,0.000
fouling_dominant,0.064,0.917,0.000,0.019
decay_dominant,0.000,0.000,0.992,0.008
combined,0.032,0.108,0.094,0.766


## 5. Saturation split (L1): the same three numbers for unsaturated and saturated windows

In [6]:
satfrac = (dL["x"][:, :, 3] >= 395).mean(axis=1); sat = satfrac > 0.5
parts = []
for lab, mk in (("unsaturated", ~sat), ("saturated (>50% of samples at Qc >= 395)", sat)):
    t = three(smpL, thL, nL, mask=mk); t.insert(0, "group", lab); parts.append(t)
split = pd.concat(parts); display(split.round(3)); split.to_csv(R / "11_N3_L1_saturation_split.csv", index=False)
print("fraction of saturated windows per scenario:", {s: round(float(sat[nL == s].mean()), 3) for s in sorted(set(nL))})


,group,scenario,n,class_acc,acc_se,cov90_b,cov_se,sd_b,bias_b,sd_a
0,unsaturated,Sc1,784,0.980,0.005,0.918,0.010,0.040,0.001,0.055
1,unsaturated,Sc2,610,0.870,0.014,0.900,0.012,0.038,0.004,0.055
2,unsaturated,Sc3,822,0.998,0.002,0.923,0.009,0.042,0.005,0.029
3,unsaturated,Sc4,605,0.779,0.017,0.907,0.012,0.038,0.004,0.043
4,unsaturated,Sc5,210,0.976,0.011,0.948,0.015,0.036,0.005,0.054
5,unsaturated,Sc7,599,0.456,0.020,0.003,0.002,0.029,-0.390,0.080
0,saturated (>50% of samples at Qc >= 395),Sc1,216,0.935,0.017,0.907,0.020,0.056,-0.017,0.057
1,saturated (>50% of samples at Qc >= 395),Sc2,390,0.836,0.019,0.910,0.014,0.054,-0.010,0.058
2,saturated (>50% of samples at Qc >= 395),Sc3,178,0.966,0.014,0.933,0.019,0.058,-0.012,0.030
3,saturated (>50% of samples at Qc >= 395),Sc4,395,0.747,0.022,0.914,0.014,0.055,-0.003,0.045


fraction of saturated windows per scenario: {np.str_('Sc1'): 0.216, np.str_('Sc2'): 0.39, np.str_('Sc3'): 0.178, np.str_('Sc4'): 0.395, np.str_('Sc5'): 0.79, np.str_('Sc7'): 0.401}


## 6. Sensitivity: the naive posterior (nominal inlet assumed) on D1 and on D0

In [7]:
post_pn, prod_n = P3.load_naive_posterior()
print("Naive posterior:", prod_n["production_budget"], "simulations, ensemble of %d" % len(prod_n["members"]), "| summary set", prod_n["summary_set"])
smp_n1, _ = evaluate(post_pn, P3.naive_features(d1), th1, n1)
naive_d1 = three(smp_n1, th1, n1); naive_d1.insert(0, "data", "D1 (inlet varies, naive posterior assumes nominal)")
d0 = P3.load_d0(); cl = np.isin(d0["scenario_id"], [1, 2, 3, 4, 5, 7])
d0c = {k: (v[cl] if hasattr(v, "shape") and v.shape[:1] == cl.shape else v) for k, v in d0.items()}
n0 = names_of(d0c); th0 = d0c["theta"][:, 2:]
smp_n0, _ = evaluate(post_pn, P3.naive_features(d0c), th0, n0)
naive_d0 = three(smp_n0, th0, n0); naive_d0.insert(0, "data", "D0 (nominal inlet, naive posterior valid)")
sens = pd.concat([naive_d0, naive_d1]); display(sens.round(3)); sens.to_csv(R / "11_N3_naive_sensitivity.csv", index=False)
print(f"notebook wall time {(time.time() - T0) / 60:.1f} min")


Naive posterior: 40000 simulations, ensemble of 5 | summary set S1_base_est


,data,scenario,n,class_acc,acc_se,cov90_b,cov_se,sd_b,bias_b,sd_a
0,"D0 (nominal inlet, naive posterior valid)",Sc1,50,1.00,0.000,0.94,0.034,0.015,0.001,0.053
1,"D0 (nominal inlet, naive posterior valid)",Sc2,50,0.98,0.020,0.96,0.028,0.014,-0.002,0.052
2,"D0 (nominal inlet, naive posterior valid)",Sc3,50,1.00,0.000,0.94,0.034,0.016,0.002,0.026
3,"D0 (nominal inlet, naive posterior valid)",Sc4,50,0.94,0.034,0.88,0.046,0.014,-0.000,0.038
4,"D0 (nominal inlet, naive posterior valid)",Sc5,50,0.98,0.020,0.90,0.042,0.024,0.001,0.057
5,"D0 (nominal inlet, naive posterior valid)",Sc7,50,0.32,0.066,0.00,0.000,0.020,-0.383,0.062
0,"D1 (inlet varies, naive posterior assumes nomi...",Sc1,50,0.58,0.070,0.08,0.038,0.018,-0.086,0.051
1,"D1 (inlet varies, naive posterior assumes nomi...",Sc2,50,0.24,0.060,0.04,0.028,0.014,0.021,0.052
2,"D1 (inlet varies, naive posterior assumes nomi...",Sc3,50,0.52,0.071,0.04,0.028,0.020,-0.092,0.025
3,"D1 (inlet varies, naive posterior assumes nomi...",Sc4,50,0.36,0.068,0.02,0.020,0.020,0.083,0.044


notebook wall time 3.7 min


## 6b. Interpretation

**What the production posterior delivers (L1, 1,000 windows per scenario; D1's 50 per scenario agree within their standard errors).**

| | Sc1 | Sc2 | Sc3 | Sc4 | Sc5 | Sc7 |
|---|---|---|---|---|---|---|
| class accuracy | 0.970 | 0.857 | 0.992 | 0.766 | 0.977 | 0.341 |
| β 90% coverage | 0.916 | 0.904 | 0.925 | 0.910 | 0.927 | 0.002 |
| β posterior sd | 0.044 | 0.044 | 0.045 | 0.045 | 0.043 | 0.029 |

Macro-F1 over the closed-loop scenarios Sc1–Sc5 (headline) is **0.907** (accuracy 0.912); including the Sc7 sensor-drift scenario it is 0.824.

**Reliable half of the storyline.** In Sc1–Sc5 the β posterior is calibrated where it is tested: coverage 0.90–0.93 against the nominal 0.90, bias |β̂ − β| ≤ 0.004,
and the same holds in the unsaturated and the saturated windows separately (coverage 0.90–0.95 and 0.91–0.93). The posterior is therefore
honest about its width, and the width is what the sensors and the inlet allow: sd(β) ≈ 0.044 is set by the 0.1 K calibration offset of the inlet
measurements (notebooks 34–36), not by the network. It is not a result about the method being sharp.

**What limits classification.** Accuracy is high (≥ 0.97) for Sc1, Sc3 and Sc5, and lower for Sc2 (0.857) and Sc4 (0.766). The confusion matrix shows the
errors are neighbour confusions across the class thresholds: 127 fouling-dominant windows read as healthy, 108 combined windows read as fouling-dominant and 94 as
decay-dominant. With a posterior sd of 0.044 in β (and 0.03–0.06 in α), a scenario whose true parameters lie within about one posterior sd of a class
threshold cannot be classified more reliably than this; that is a property of the information in the window, not a defect to be tuned away. We therefore
report the three numbers per scenario together (accuracy, coverage, sd) rather than accuracy alone, and the classification step is read together with
the posterior width.

**Saturated windows.** With the inlet varying by ±2 K the β = 0.85 scenarios hit the jacket limit in a large share of windows (Sc5: 790 of 1,000 windows with
more than half of the samples at Qc ≥ 395 L/min). Coverage and bias are unchanged there; the β sd is wider (0.045–0.058 against 0.036–0.042 unsaturated),
which is the expected loss of information when the valve is clipped. Saturation costs width, not calibration.

**Silent failure documented: Sc7 (unmodelled +2 K temperature-sensor drift).** The posterior is confidently wrong: β̂ is biased by −0.395 with sd 0.029, 90% coverage is 0.002,
and 66% of the windows are classed as *combined* (34% as fouling-dominant, the true class; 0.1% healthy), with mean α̂ = 0.885 against a true 1.00. Nothing in the
posterior width signals this. We have not isolated the mechanism here (the closed loop shifts the true operating point to compensate the biased reading, and the posterior reads the shifted jacket response as extra fouling); notebook 09 tests whether modelling the drift removes it.
This is the paper's sensor-specification message in one row: the sensor error that the simulator does not model is not seen by SBI, and the output looks healthy.
Sc7 is reported in the main text only if the drift-aware model of notebook 09 (Phase 3 step f) fixes it; otherwise it stays here as the documented failure mode.

**Inlet silent failure on the paper's own data (sensitivity only, not a headline).** The *naive* posterior (nominal inlet assumed, notebook 31) is excellent on the
nominal-inlet dataset D0 (β coverage 0.88–0.96, accuracy 0.94–1.00 for Sc1–Sc5) and collapses on D1, where the inlet varies within ±2 K: coverage 0.02–0.08 and accuracy 0.24–0.58.
The same posterior, the same sensors, the same scenarios: only the inlet assumption changed. This is why the production model of notebook 36 conditions on the measured inlet,
and it is the second documented silent failure (control law, warm start, sensor model and inlet conditions being the others in the storyline). The production posterior's results above
are the remedy's performance; this section is the evidence that the remedy was needed.

**Ledger.** Supports: Macro-F1 over closed-loop scenarios (0.907, Sc1–Sc5) and the per-scenario three numbers (Table 7 rows). Documents a failure mode: Sc7; naive-on-D1.
Does not support: any claim that classification is near-perfect in Sc2/Sc4, or that β is identified more sharply than ≈ 0.044 under N3 sensors and a 0.1 K inlet calibration offset.